# Libs

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots


import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.MSCVAE import MSCVAE
from src.MSCRED import MSCRED
from src.ANN_AE import ANN_AE
from src.LSTM_AE import LSTM_AE
from src.CNN_AE import CNN_AE
from src.USAD import USAD
from src.OmniAnomaly import OmniAnomaly
from src.TranAD import TranAD

DIR_DATA = os.getcwd()+"/data/TE_data/"

# Load Data

In [ ]:
import scipy.io as sio

arquivos_mat = glob.glob(DIR_DATA + "*.mat")

for caminho in arquivos_mat:
    nome_base = os.path.splitext(os.path.basename(caminho))[0]
    data = sio.loadmat(caminho)
    
    if "fault_00" in nome_base:
        globals()["df_normal"] = pd.DataFrame(data["trainingdata"])
        globals()[f"df_{nome_base}"] = pd.DataFrame(data["testdata"])
    else:
        globals()[f"df_{nome_base}"] = pd.DataFrame(data["testdata"])

In [ ]:
df_sistema = pd.DataFrame({
    "VARIAVEL": list(range(52)),
    "DESC": [f"Var {i}" for i in range(52)],
    "SISTEMA": ["TE"] * 52
})
df_sistema

# Fit

In [ ]:
mscvae = MSCVAE()

gain = 1
epochs = 100
mscvae.fit([df_normal], gain=gain, epochs=epochs, verbose=True)

# Predict

In [ ]:
df_falha = df_fault_12

predictions = mscvae.predict(df_falha)

In [ ]:
def plot_predict(predictions, threshold):
    # Criar o dataframe extraindo diretamente da sua estrutura de saída
    df = pd.DataFrame({
        "timestamp": predictions['timestamp'],
        "phi": predictions['phi']/threshold
    })

    df["threshold"] = threshold/threshold

    # Configurar o layout
    layout = go.Layout(plot_bgcolor="white", paper_bgcolor='white')
    fig = go.Figure(layout=layout)
    
    # Adicionar a série do Índice (Phi)
    fig.add_trace(go.Scatter(
        x=df["timestamp"], 
        y=df["phi"], 
        mode="lines", 
        name="Índice (Phi)", 
        fill="tozeroy", 
        line_color="#0F293A"
    ))
    
    # Adicionar a linha do Limiar
    fig.add_trace(go.Scatter(
        x=df["timestamp"], 
        y=df["threshold"], 
        mode="lines", 
        name="Limiar", 
        line_color="#FB8102"
    ))
    
    # Ajustar eixos
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor='#CECFD1')
    
    # Ajustar layout final
    fig.update_layout(
        hovermode='x unified', 
        legend=dict(orientation="h"),
        yaxis_range=[0, 4 * threshold/threshold] # Mantém o zoom vertical focado no limite inferior
    )

    return fig

fig = plot_predict(predictions, threshold=mscvae.threshold)
fig.show()

# Contribution

In [ ]:
contribution, df_reconstruction = mscvae.contribution(df_falha, df_sistema, top_k=None)
df_contribution = pd.DataFrame().from_dict(contribution).head(10)
df_contribution

# Plot variables

In [ ]:
def plot_reconstruction(df_original, df_reconstruction, vars_to_plot):
    num_vars = len(vars_to_plot)
    
    if num_vars == 0:
        return "Nenhuma variável selecionada."

    # Cria subplots dinamicamente com base na quantidade de variáveis
    fig = make_subplots(
        rows=num_vars, 
        cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.05,
        subplot_titles=[f"Variável: {var}" for var in vars_to_plot]
    )
    
    for i, var in enumerate(vars_to_plot):
        row_idx = i + 1
        
        # Linha da variável original
        fig.add_trace(go.Scatter(
            x=df_original.index, 
            y=df_original[var], 
            mode="lines", 
            line_color="#1f77b4"
        ), row=row_idx, col=1)
        
        # Linha da variável reconstruída
        fig.add_trace(go.Scatter(
            x=df_reconstruction.index, 
            y=df_reconstruction[var], 
            mode="lines", 
            line_color="#d62728", 
            line_dash="dot"
        ), row=row_idx, col=1)

    # Configurações de layout
    fig.update_layout(
        plot_bgcolor="white", 
        paper_bgcolor='white',
        hovermode='x unified',
        height=250 * num_vars, # Ajusta a altura de acordo com a quantidade de gráficos
    )
    
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor='#CECFD1')
    
    return fig

In [ ]:
# Passando a lista de variáveis que deseja inspecionar
variaveis = list(df_contribution['VARIAVEL'])

fig_recon = plot_reconstruction(df_falha, df_reconstruction, variaveis)
fig_recon.show()